In [ ]:
from google.colab import files
uploaded = files.upload()


Saving comentarios_experimento_youtube.csv to comentarios_experimento_youtube.csv


In [ ]:
!pip install -q pysentimiento pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 21.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re

df = pd.read_csv("comentarios_experimento_youtube.csv")
df.head()

texto_col = "text"


In [ ]:
def limpiar_texto(t):
    t = str(t)
    t = re.sub(r"http\S+", " ", t)      # URLs
    t = re.sub(r"@\w+", " ", t)        # menciones (por si las hay)
    t = re.sub(r"#", " ", t)           # #
    t = re.sub(r"\s+", " ", t).strip()
    return t

df = df.dropna(subset=[texto_col])
df = df.drop_duplicates(subset=[texto_col])

df["texto_limpio"] = df[texto_col].apply(limpiar_texto)
df[["texto_limpio"]].head()


,texto_limpio
0,Este fue uno de los mejores reportajes de Juan...
1,"""Tranquilos los organismos de control investig..."
2,"Buenos días,las palabras se las lleva el vient..."
3,El problema está en nombrar gobernantes inepto...
4,"Realmente en este gobierno de ""carruseles de c..."


In [ ]:
from pysentimiento import create_analyzer

# Modelo de sentimiento en español (pos/neg/neu) basado en BETO
analyzer = create_analyzer(task="sentiment", lang="es")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

In [ ]:
def analizar_sentimiento(texto):
    res = analyzer.predict(texto)
    return pd.Series({
        "label": res.output,                 # 'POS', 'NEG', 'NEU'
        "proba_pos": res.probas.get("POS", 0.0),
        "proba_neg": res.probas.get("NEG", 0.0),
        "proba_neu": res.probas.get("NEU", 0.0),
    })

resultados = df["texto_limpio"].apply(analizar_sentimiento)

df_beto = pd.concat([df, resultados], axis=1)
df_beto.head()


,video_url,author,text,time,votes,texto_limpio,label,proba_pos,proba_neg,proba_neu
0,https://www.youtube.com/watch?v=yaA1_92D_Lo,@senorx6845,Este fue uno de los mejores reportajes de Jua...,hace 4 años,62,Este fue uno de los mejores reportajes de Juan...,POS,0.967160,0.004046,0.028793
1,https://www.youtube.com/watch?v=yaA1_92D_Lo,@JesusMartinez-ru5qk,"""Tranquilos los organismos de control investig...",hace 4 años (editado),33,"""Tranquilos los organismos de control investig...",NEU,0.179014,0.134064,0.686922
2,https://www.youtube.com/watch?v=yaA1_92D_Lo,@juancarlosjaramillo2967,"Buenos días,las palabras se las lleva el vient...",hace 4 años,22,"Buenos días,las palabras se las lleva el vient...",NEG,0.012860,0.933124,0.054016
3,https://www.youtube.com/watch?v=yaA1_92D_Lo,@matildesuarez1708,El problema está en nombrar gobernantes inepto...,hace 4 años,13,El problema está en nombrar gobernantes inepto...,NEG,0.006392,0.941198,0.052410
4,https://www.youtube.com/watch?v=yaA1_92D_Lo,@mariomeneses7231,"Realmente en este gobierno de ""carruseles de c...",hace 4 años,13,"Realmente en este gobierno de ""carruseles de c...",NEG,0.005098,0.947305,0.047597


In [ ]:
df_beto["label"].value_counts()


,count
label,
NEG,1794
NEU,806
POS,391


In [ ]:
print("🔴Ejemplos NEGATIVOS")
df_beto[df_beto["label"] == "NEG"][["texto_limpio", "proba_neg"]].head(10)

print("🟢Ejemplos POSITIVOS")
df_beto[df_beto["label"] == "POS"][["texto_limpio", "proba_pos"]].head(10)

print("⚪️Ejemplos NEUTROS")
df_beto[df_beto["label"] == "NEU"][["texto_limpio", "proba_neu"]].head(10)


🔴Ejemplos NEGATIVOS
🟢Ejemplos POSITIVOS
⚪️Ejemplos NEUTROS


,texto_limpio,proba_neu
1,"""Tranquilos los organismos de control investig...",0.686922
11,"Solo hay que cambiar a todos los gobernantes, ...",0.742938
13,No estaba muerta andaba de parranda...🎶,0.798601
21,"Cuál Juan Diego, todo lo qie sabe es gracias P...",0.480840
22,Jajajaja,0.500932
25,👏👏👏👏👏👏,0.571618
28,"Polombia es el único lugar del mundo, donde Co...",0.509298
30,Juan Diego es un despistado tenía que hacer es...,0.579364
31,"Leyes y mas leyes, dependencias y órganos de c...",0.655363
32,Hahahaha la Honorable ya debe haber cambiado d...,0.617839


In [ ]:
df_beto.to_csv("comentarios_experimento_youtube_beto.csv", index=False)

from google.colab import files
files.download("comentarios_experimento_youtube_beto.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Notebook de la descarga de los datos: https://colab.research.google.com/drive/1dzZmis_AmT-xwanq0JG82cVV1JzpTmGo?usp=sharing